In [1]:
!pip install torch-geometric
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.1.0+cu121.html


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.0 MB/s eta 0:00:0000:01
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=640889 sha256=4da3004f4cdd1677acd67ce285b23dd549b62e142214d8623f4fac6b64dc33b8
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
Successfully built torch-scatter


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_scatter import scatter
from torch_geometric.utils import softmax as pyg_softmax
import timeit
import os
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# dummy dataset 

In [3]:

from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np

class DTADataset(Dataset):
    """Real Drug-Target Affinity Dataset"""
    
    def __init__(self, csv_file, drugs_dir='/kaggle/input/davisd/kaggle/drugs', 
                 proteins_dir='/kaggle/input/davisd/kaggle/proteins',
                 embeddings_dir='/kaggle/input/davisd/kaggle/ESM_embedding_pocket'):
        self.df = pd.read_csv(csv_file)
        self.drugs_dir = drugs_dir
        self.proteins_dir = proteins_dir
        self.embeddings_dir = embeddings_dir
        print(f"✓ Loaded {len(self.df)} drug-protein pairs")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        compound_id = row['COMPOUND_ID']
        protein_id = row['PROTEIN_ID']
        label = float(row['REG_LABEL'])
        
        # Load drug graph (PyG Data object)
        drug_path = os.path.join(self.drugs_dir, f'{compound_id}.pt')
        # drug_graph = torch.load(drug_path)
        drug_graph = torch.load(drug_path, weights_only=False)

        # Load protein tensor dict
        protein_path = os.path.join(self.proteins_dir, f'{protein_id}.pt')
        # protein_dict = torch.load(protein_path)
        protein_dict = torch.load(protein_path, weights_only=False)

        # Wrap in PyG Data object
        protein_graph = Data(
            x=protein_dict['x'],
            edge_index=protein_dict['edge_index'],
            edge_attr=protein_dict['edge_attr'],
            laplacian_pe=protein_dict['laplacian_pe']
        )
        
        # Load ESM embedding
        esm_path = os.path.join(self.embeddings_dir, f'{protein_id}.npy')
        protein_embedding = np.load(esm_path, allow_pickle=True)
        
        return drug_graph, protein_graph, protein_embedding, label


def custom_collate_dta(batch):
    """Collate function for real dataset"""
    drug_graphs = [item[0] for item in batch]
    protein_graphs = [item[1] for item in batch]
    protein_embeddings = [item[2] for item in batch]
    labels = [item[3] for item in batch]
    
    # Batch graphs
    drug_batch = Batch.from_data_list(drug_graphs)
    protein_batch = Batch.from_data_list(protein_graphs)
    
    # Pad embeddings
    max_len = max(emb.shape[0] for emb in protein_embeddings)
    padded = []
    for emb in protein_embeddings:
        if emb.shape[0] < max_len:
            pad_size = max_len - emb.shape[0]
            padded.append(np.pad(emb, ((0, pad_size), (0, 0)), 
                                mode='constant', constant_values=0))
        else:
            padded.append(emb)
    
    protein_embedding_batch = torch.FloatTensor(np.array(padded))
    label_batch = torch.FloatTensor(labels)
    
    return drug_batch, protein_batch, protein_embedding_batch, label_batch

In [5]:
dataset = DTADataset(
        csv_file='/kaggle/input/davisd/kaggle/Davis.csv'
    )
loader = DataLoader(dataset, batch_size=4, collate_fn=custom_collate_dta)

drug_batch, protein_batch, protein_embedding_batch, label_batch = next(iter(loader))

print("\n===== Final Output From Loader =====")
print("Drug batch:", drug_batch)
print("Protein batch:", protein_batch)
print("Protein embedding batch shape:", protein_embedding_batch.shape)
print("Label batch shape:", label_batch.shape)


✓ Loaded 24956 drug-protein pairs

===== Final Output From Loader =====
Drug batch: DataBatch(x=[108, 44], edge_index=[2, 240], edge_attr=[240, 10], laplacian_pe=[108, 8], smiles=[4], batch=[108], ptr=[5])
Protein batch: DataBatch(x=[66, 41], edge_index=[2, 302], edge_attr=[302, 5], laplacian_pe=[66, 8], batch=[66], ptr=[5])
Protein embedding batch shape: torch.Size([4, 27, 320])
Label batch shape: torch.Size([4])


# graph_transformer_edge_layer

In [19]:
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, use_bias):
        super().__init__()
        
        self.out_dim = out_dim
        self.num_heads = num_heads
        
        if use_bias:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=True)
        else:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=False)
    
    def forward(self, h, edge_index, e):
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # Project to Q, K, V
        Q_h = self.Q(h).view(num_nodes, self.num_heads, self.out_dim)
        K_h = self.K(h).view(num_nodes, self.num_heads, self.out_dim)
        V_h = self.V(h).view(num_nodes, self.num_heads, self.out_dim)
        proj_e = self.proj_e(e).view(num_edges, self.num_heads, self.out_dim)
        
        # Get source and target node indices
        src, dst = edge_index[0], edge_index[1]
        
        # Gather K, Q, V for each edge
        K_j = K_h[src]  # [num_edges, num_heads, out_dim]
        Q_i = Q_h[dst]  # [num_edges, num_heads, out_dim]
        V_j = V_h[src]  # [num_edges, num_heads, out_dim]
        
        # Compute attention scores: Q·K / sqrt(d_k)
        score = (Q_i * K_j) / (self.out_dim ** 0.5)  # [num_edges, num_heads, out_dim]
        
        # Modulate by edge features (element-wise multiplication)
        score = score * proj_e  # [num_edges, num_heads, out_dim]
        
        # Store edge output (before softmax)
        e_out = score
        
        # Sum across feature dimension for softmax
        score_scalar = score.sum(dim=-1)  # [num_edges, num_heads]
        
        # Clamp for numerical stability
        score_scalar = score_scalar.clamp(-5, 5)
        
        # Apply softmax per target node for each head separately
        attn_weights = torch.zeros_like(score_scalar)
        for head in range(self.num_heads):
            attn_weights[:, head] = pyg_softmax(score_scalar[:, head], dst, num_nodes=num_nodes)
        
        # Expand for multiplication with V
        attn = attn_weights.unsqueeze(-1)  # [num_edges, num_heads, 1]
        
        # Weight the values
        msg = V_j * attn  # [num_edges, num_heads, out_dim]
        
        # Aggregate messages to each node
        h_out = scatter(msg, dst, dim=0, dim_size=num_nodes, reduce='sum')
        
        # Compute normalization factor (sum of attention weights)
        z = scatter(attn, dst, dim=0, dim_size=num_nodes, reduce='sum')
        
        # Normalize
        h_out = h_out / (z + 1e-6)
        
        return h_out, e_out

In [20]:
class GraphTransformerLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, dropout, layer_norm=True, batch_norm=False, residual=True, use_bias=False):
        super().__init__()
        self.in_channels = in_dim
        self.out_channels = out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.residual = residual
        self.layer_norm = layer_norm
        self.batch_norm = batch_norm
        
        self.attention = MultiHeadAttentionLayer(in_dim, out_dim // num_heads, num_heads, use_bias)
        
        self.O_h = nn.Linear(out_dim, out_dim)
        self.O_e = nn.Linear(out_dim, out_dim)
        
        if self.layer_norm:
            self.layer_norm1_h = nn.LayerNorm(out_dim)
            self.layer_norm1_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm1_h = nn.BatchNorm1d(out_dim)
            self.batch_norm1_e = nn.BatchNorm1d(out_dim)
        
        self.FFN_h_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_h_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        self.FFN_e_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_e_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        if self.layer_norm:
            self.layer_norm2_h = nn.LayerNorm(out_dim)
            self.layer_norm2_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm2_h = nn.BatchNorm1d(out_dim)
            self.batch_norm2_e = nn.BatchNorm1d(out_dim)
    
    def forward(self, h, edge_index, e):
        h_in1 = h
        e_in1 = e
        
        # Multi-head attention
        h_attn_out, e_attn_out = self.attention(h, edge_index, e)
        
        h = h_attn_out.view(-1, self.out_channels)
        e = e_attn_out.view(-1, self.out_channels)
        
        h = F.dropout(h, self.dropout, training=self.training)
        e = F.dropout(e, self.dropout, training=self.training)
        
        h = self.O_h(h)
        e = self.O_e(e)
        
        if self.residual:
            h = h_in1 + h
            e = e_in1 + e
        
        if self.layer_norm:
            h = self.layer_norm1_h(h)
            e = self.layer_norm1_e(e)
        
        if self.batch_norm:
            h = self.batch_norm1_h(h)
            e = self.batch_norm1_e(e)
        
        h_in2 = h
        e_in2 = e
        
        # FFN for h
        h = self.FFN_h_layer1(h)
        h = F.relu(h)
        h = F.dropout(h, self.dropout, training=self.training)
        h = self.FFN_h_layer2(h)
        
        # FFN for e
        e = self.FFN_e_layer1(e)
        e = F.relu(e)
        e = F.dropout(e, self.dropout, training=self.training)
        e = self.FFN_e_layer2(e)
        
        if self.residual:
            h = h_in2 + h
            e = e_in2 + e
        
        if self.layer_norm:
            h = self.layer_norm2_h(h)
            e = self.layer_norm2_e(e)
        
        if self.batch_norm:
            h = self.batch_norm2_h(h)
            e = self.batch_norm2_e(e)
        
        return h, e

In [21]:
class GraphTransformerCompound(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        # print("here ")
        h = data.x.float().to(self.device)
        # print(h.shape)
        h_lap_pos_enc = data.laplacian_pe.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h
    

In [22]:
class GraphTransformerProtein(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        h = data.x.float().to(self.device)
        h_lap_pos_enc = data.laplacian_pe.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h

In [23]:
class SelfAttention(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout):
        super().__init__()
        
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        assert hid_dim % n_heads == 0
        
        self.w_q = nn.Linear(hid_dim, hid_dim)
        self.w_k = nn.Linear(hid_dim, hid_dim)
        self.w_v = nn.Linear(hid_dim, hid_dim)
        self.fc = nn.Linear(hid_dim, hid_dim)
        self.do = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([hid_dim // n_heads])).to(device)
    
    def forward(self, query, key, value, mask=None):
        bsz = query.shape[0]
        
        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)
        
        Q = Q.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        K = K.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        V = V.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        
        attention = self.do(F.softmax(energy, dim=-1))
        x = torch.matmul(attention, V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, -1, self.n_heads * (self.hid_dim // self.n_heads))
        x = self.fc(x)
        
        return x

In [24]:
def pyg_batch_split(bg, feats):  # Even named 'bg' like DGL!
    """
    Exact DGL logic, just swapping data accessors
    """
    # Only difference: PyG needs to compute batch_num_nodes
    batch = bg.batch
    batch_num_nodes = scatter(
        torch.ones_like(batch), batch, 
        dim=0, dim_size=bg.num_graphs, reduce='sum'
    )
    
    # Now follow DGL exactly
    max_num_nodes = int(batch_num_nodes.max())
    cum_nodes = torch.cat([batch.new_zeros(1), batch_num_nodes.cumsum(dim=0)])
    idx = torch.arange(feats.size(0), dtype=torch.long, device=feats.device)
    idx = (idx - cum_nodes[batch]) + (batch * max_num_nodes)
    size = [bg.num_graphs * max_num_nodes] + list(feats.size())[1:]
    out = feats.new_full(size, fill_value=0)
    out[idx] = feats
    out = out.view([bg.num_graphs, max_num_nodes] + list(feats.size())[1:])
    return out

In [25]:
class MDGTDTInet(nn.Module):
    def __init__(self, compound_dim=128, protein_dim=128, gt_layers=10, gt_heads=8, out_dim=1):
        super(MDGTDTInet, self).__init__()
        self.compound_dim = compound_dim
        self.protein_dim = protein_dim
        self.n_layers = gt_layers
        self.n_heads = gt_heads
        
        self.crossAttention = SelfAttention(hid_dim=self.compound_dim, n_heads=1, dropout=0.2)
        
        self.compound_gt = GraphTransformerCompound(
            device, n_layers=gt_layers, node_dim=44, edge_dim=10, 
            hidden_dim=compound_dim, out_dim=compound_dim, n_heads=gt_heads, 
            in_feat_dropout=0.0, dropout=0.2, pos_enc_dim=8
        )
        
        self.protein_gt = GraphTransformerProtein(
            device, n_layers=gt_layers, node_dim=41, edge_dim=5, 
            hidden_dim=protein_dim, out_dim=protein_dim, n_heads=gt_heads, 
            in_feat_dropout=0.0, dropout=0.2, pos_enc_dim=8
        )
        
        self.protein_embedding_fc = nn.Linear(320, self.protein_dim)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
        self.joint_attn_prot = nn.Linear(compound_dim, compound_dim)
        self.joint_attn_comp = nn.Linear(compound_dim, compound_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(compound_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, out_dim)
        )
    
    def forward(self, compound_graph, protein_graph, protein_embedding):
        # Compound encoding
        compound_feat = self.compound_gt(compound_graph)
        compound_feat_x = pyg_batch_split(compound_graph, compound_feat)
        compound_feats = compound_feat_x
        
        # Protein encoding
        protein_feat = self.protein_gt(protein_graph)
        protein_feat_x = pyg_batch_split(protein_graph, protein_feat)
        protein_embedding = self.protein_embedding_fc(protein_embedding)
        protein_feats = self.crossAttention(protein_embedding, protein_feat_x, protein_feat_x)
        
        # Compound-protein interaction
        inter_comp_prot = self.sigmoid(
            torch.einsum('bij,bkj->bik', 
                        self.joint_attn_prot(self.relu(protein_feats)), 
                        self.joint_attn_comp(self.relu(compound_feats)))
        )
        inter_comp_prot_sum = torch.einsum('bij->b', inter_comp_prot)
        inter_comp_prot = torch.einsum('bij,b->bij', inter_comp_prot, 1/inter_comp_prot_sum)
        
        # Compound-protein joint embedding
        cp_embedding = self.tanh(torch.einsum('bij,bkj->bikj', protein_feats, compound_feats))
        cp_embedding = torch.einsum('bijk,bij->bk', cp_embedding, inter_comp_prot)
        
        x = self.classifier(cp_embedding)
        
        return x

In [26]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def ci(y_true, y_pred):
    """Concordance Index"""
    ind = np.argsort(y_true)
    y_true = y_true[ind]
    y_pred = y_pred[ind]
    i = len(y_true) - 1
    j = i - 1
    z = 0.0
    S = 0.0
    while i > 0:
        while j >= 0:
            if y_true[i] > y_true[j]:
                z = z + 1
                u = y_pred[i] - y_pred[j]
                if u > 0:
                    S = S + 1
                elif u == 0:
                    S = S + 0.5
            j = j - 1
        i = i - 1
        j = i - 1
    ci = S / z
    return ci

def rm2(y_true, y_pred):
    """R squared metric"""
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)
    r02 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true ** 2))
    return r2 if r2 > r02 else r02

In [27]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()
    for batch_idx, data in enumerate(train_loader):
        compound_graph, protein_graph, protein_embedding, label = data
        
        compound_graph = compound_graph.to(device)
        protein_graph = protein_graph.to(device)
        protein_embedding = protein_embedding.to(device)
        label = label.to(device)
        
        output = model(compound_graph, protein_graph, protein_embedding)
        loss = criterion(output, label.view(-1, 1).float())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def test(model, device, test_loader):
    model.eval()
    total_preds = torch.Tensor()
    total_labels = torch.Tensor()
    
    with torch.no_grad():
        for data in test_loader:
            compound_graph, protein_graph, protein_embedding, label = data
            
            compound_graph = compound_graph.to(device)
            protein_graph = protein_graph.to(device)
            protein_embedding = protein_embedding.to(device)
            
            output = model(compound_graph, protein_graph, protein_embedding)
            total_preds = torch.cat((total_preds, output.cpu()), 0)
            total_labels = torch.cat((total_labels, label.view(-1, 1).cpu()), 0)
    
    total_labels = total_labels.numpy().flatten()
    total_preds = total_preds.numpy().flatten()
    
    MSE = mse(total_labels, total_preds)
    RMSE = rmse(total_labels, total_preds)
    CI = ci(total_labels, total_preds)
    RM2 = rm2(total_labels, total_preds)
    
    return MSE, RMSE, CI, RM2

In [ ]:
# ============================================================================
# MAIN TRAINING - REPLACE __main__ SECTION
# ============================================================================

if __name__ == '__main__':
    import timeit
    
    print("="*70)
    print("🚀 AttentionMGT-DTA Training (REAL DATASET)")
    print("="*70)
    
    # ✅ Create REAL dataset
    dataset = DTADataset(
        csv_file='/kaggle/input/davisd/kaggle/Davis.csv'
    )
    
    # Split train/test (80/20)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, test_size]
    )
    
    print(f"\n📊 Dataset Split:")
    print(f"  Training:   {len(train_dataset)} samples")
    print(f"  Testing:    {len(test_dataset)} samples")
    
    # DataLoaders
    batch_size = 32
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=custom_collate_dta,  # ✅ Use real collate
        num_workers=0,
        drop_last=True
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=custom_collate_dta,  # ✅ Use real collate
        num_workers=0,
        drop_last=True
    )
    
    print(f"\n📦 DataLoaders:")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Test batches:  {len(test_loader)}")
    print(f"  Batch size:    {batch_size}")
    
    # Model
    print(f"\n🏗️ Building Model...")
    model = MDGTDTInet(
        compound_dim=128, 
        protein_dim=128, 
        gt_layers=10, 
        gt_heads=8, 
        out_dim=1
    )
    model.to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_params:,}")
    
    # Training setup
    lr = 1e-4
    epochs = 50
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer, mode='min', factor=0.8, 
        patience=10, min_lr=1e-5
    )
    criterion = nn.MSELoss()
    
    print(f"\n⚙️ Training Config:")
    print(f"  Learning rate: {lr}")
    print(f"  Epochs:        {epochs}")
    print(f"  Device:        {device}")
    
    # Training loop
    print(f"\n{'='*70}")
    print("🔥 Starting Training...")
    print(f"{'='*70}")
    print('Epoch\t\tTime\t\tMSE\t\tRMSE\t\tCI\t\tRM2')
    print('-'*70)
    
    start = timeit.default_timer()
    best_mse = 100
    best_epoch = -1
    
    for epoch in range(epochs):
        # Train
        train(model, device, train_loader, optimizer, criterion)
        
        # Test
        mse_test, rmse_test, ci_test, rm2_test = test(model, device, test_loader)
        
        # Scheduler step
        scheduler.step(mse_test)
        
        # Time
        end = timeit.default_timer()
        time = end - start
        
        # Print results
        ret = [epoch + 1, round(time, 2), round(mse_test, 5), 
               round(rmse_test, 5), round(ci_test, 5), round(rm2_test, 5)]
        print('\t\t'.join(map(str, ret)))
        
        # Save best model
        if mse_test < best_mse:
            best_epoch = epoch + 1
            best_mse = mse_test
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'  ✓ MSE improved! Saved model (epoch {best_epoch}, MSE: {best_mse:.5f})')
    
    print(f"\n{'='*70}")
    print(f"✅ Training Complete!")
    print(f"{'='*70}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best MSE:   {best_mse:.5f}")
    print(f"{'='*70}")

🚀 AttentionMGT-DTA Training (REAL DATASET)
✓ Loaded 24956 drug-protein pairs

📊 Dataset Split:
  Training:   19964 samples
  Testing:    4992 samples

📦 DataLoaders:
  Train batches: 623
  Test batches:  156
  Batch size:    32

🏗️ Building Model...
  Total parameters: 6,228,609

⚙️ Training Config:
  Learning rate: 0.0001
  Epochs:        50
  Device:        cpu

🔥 Starting Training...
Epoch		Time		MSE		RMSE		CI		RM2
----------------------------------------------------------------------
